# 채용 공고 데이터 수집, 분석 플랫폼

**목표**

사람인 채용 사이트 공고를 수집하여 기업, 기술 스택, 직무별 요구 역량, 실시간 대시보드 구축 등을 수행하는 ETL 파이프라인 구축

**아키텍처**

채용 사이트 -> Scraper -> Kafka -> ETL Worker -> Elasticsearch -> Kibana -> Dashboard

## 0. Setting

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime, timedelta
import re

from scraper import scrape, save_raw
from transformer import transform, log_data_quality, save_processed
from loader import load, LOAD_COLUMNS, MAPPING, INDEX_NAME
from kafka_producer import produce

## 1. 데이터 수집(Extract)

### 1.1 수집할 스키마 정의

```json

{
    "company_nm": "",
    "job_tit": "",
    "job_meta": [],
    "work_place": "",
    "career": "",
    "education": "",
    "salary": "",
    "deadline": "",
    "registered_at": ""
}

```

In [2]:
# 예시
{
  "company_nm": "(주)국민은행",
  "job_tit": "2026년 하계 체험형 인턴십 프로그램 「IT부문」 모집",
  "job_meta": [
    "데이터분석가",
    "데이터엔지니어",
    "정보보안",
    "정보통신"
  ],
  "work_place": "서울 영등포구",
  "career": "신입 · 인턴직",
  "education": "학력무관",
  "salary": "면접 후 협의",
  "deadline": "~06.24(수)",
  "registered_at": "3시간 전 등록"
}

{'company_nm': '(주)국민은행',
 'job_tit': '2026년 하계 체험형 인턴십 프로그램 「IT부문」 모집',
 'job_meta': ['데이터분석가', '데이터엔지니어', '정보보안', '정보통신'],
 'work_place': '서울 영등포구',
 'career': '신입 · 인턴직',
 'education': '학력무관',
 'salary': '면접 후 협의',
 'deadline': '~06.24(수)',
 'registered_at': '3시간 전 등록'}

### 1.2 HTML 구조 분석

div class = "item"

|item|list_item|col company_nm|job_tit|job_meta|col recruit_info|col support_info|
|-|---------|--------------|-------|--------|----------------|----------------|
|content|공고 하나 안에 존재|회사명|공고명|직무 태그|지역, 경력, 학력연봉|등록일, 마감일|

### 1.3 BeautifulSoup Parsing

In [3]:
url = "https://www.saramin.co.kr/zf_user/jobs/public/list?page=1&isAjaxRequest=y"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)

print(response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

jobs = soup.select("div.list_item")

def parse_job(item):

    company_nm = item.select_one(
        "div.company_nm a.str_tit"
    ).get_text(strip=True)

    job_tit = item.select_one(
        "div.job_tit a.str_tit"
    ).get_text(strip=True)

    job_meta = [
        tag.get_text(strip=True)
        for tag in item.select(
            "div.job_meta span.job_sector span"
        )
    ]

    work_place = item.select_one(
        "p.work_place"
    ).get_text(strip=True)

    career = item.select_one(
        "p.career"
    ).get_text(strip=True)

    education = item.select_one(
        "p.education"
    ).get_text(strip=True)

    salary = item.select_one(
        "p.salary"
    ).get_text(strip=True)

    deadline = item.select_one(
        "span.date"
    ).get_text(strip=True)

    registered_at = item.select_one(
        "span.deadlines"
    ).get_text(strip=True)

    return {
        "company_nm": company_nm,
        "job_tit": job_tit,
        "job_meta": job_meta,
        "work_place": work_place,
        "career": career,
        "education": education,
        "salary": salary,
        "deadline": deadline,
        "registered_at": registered_at
    }

200


In [4]:
# 가져온 채용공고 개수
jobs = soup.select("div.list_item")

print(len(jobs))

20


In [5]:
first_job = jobs[0]

print(
    first_job.select_one(
        "div.company_nm a.str_tit"
    ).get_text(strip=True)
)

(주)카카오


### 1.4 DataFrame 저장

In [6]:
results = []

for item in jobs:
    try:
        results.append(
            parse_job(item)
        )
    except:
        pass

df = pd.DataFrame(results)

In [7]:
df

,company_nm,job_tit,job_meta,work_place,career,education,salary,deadline,registered_at
0,(주)카카오,IT 정책 및 산업 동향 연구 지원_어시스턴트,"[사무직, 서무, 문서작성, 사무보조, 사무행정]",경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),27분 전 등록
1,(주)카카오,선물하기/톡딜 뷰티 CM_어시스턴트,"[이커머스, 배너광고, 배너디자인, 광고기획, 검품/검수]",경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),33분 전 등록
2,(주)카카오,쇼핑하우 가격 비교 지표 관리/정산/운영_어시스턴트,"[데이터분석가, 정산관리, 요금정산, 서비스기획, 데이터분석]",경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),38분 전 등록
3,네이버랩스(주),[네이버랩스] Manipulation Engineer,"[머신러닝, Ajax, C++, Python, Pytorch]",경기 성남시 분당구,경력무관 · 정규직,석사↑,면접 후 협의,~06.30(화),1시간 전 등록
4,(주)지학사,2026년 6월 수시채용,"[출판기획, 교재개발, 수학, 기획]",서울 마포구,신입 · 경력 · 계약직,대학교(4년)↑,면접 후 협의,~07.05(일),1시간 전 등록
5,SK텔레콤(주),퍼포먼스마케팅 디지털 광고 소재 제작 디자이너,"[AI(인공지능), AIX, 광고마케팅, 디지털마케팅, 퍼포먼스마케팅]",서울 중구,신입 · 계약직,학력무관,면접 후 협의,D-4,1시간 전 등록
6,경북대학교병원,[칠곡] 2026년 6월 1차 임시직원 채용공고(임상병리사),"[임상병리사, 병리과, 대학병원, 일반병원, 종합병원]",대구 북구,신입 · 경력 · 계약직,"대학(2,3년)↑",면접 후 협의,D-5,2시간 전 등록
7,경북대학교병원,[칠곡] 2026년 6월 1차 임시직원 채용공고(간호사),"[간호사, 수술보조, 외래, 대학병원, 일반병원]",대구 북구,신입 · 경력 · 계약직,"대학(2,3년)↑",면접 후 협의,D-5,2시간 전 등록
8,에스케이하이닉스(주),[6월] Talent hy-way 신입 (Tech R&D - 설계),"[설계엔지니어, 기술설계, 부품설계, 생산설계, 회로설계]",경기 성남시 분당구 외,신입 · 정규직,학력무관,면접 후 협의,D-6,2시간 전 등록
9,에스케이하이닉스(주),[6월] Talent hy-way 신입 (Tech R&D - 소자),"[모델링, AI(인공지능), 공정엔지니어, 공정설계, 공정관리]",경기 이천시,신입 · 정규직,학력무관,면접 후 협의,D-6,2시간 전 등록


### 1.5. 추가 컬럼

#### 1. 수집 메타데이터

In [8]:
crawl_time = datetime.now()
source = "saramin"

#### 2. 공고 URL

In [9]:
job_url = (
    "https://www.saramin.co.kr" +
    item.select_one(
        "div.job_tit a.str_tit"
    )["href"]
)

job_url

'https://www.saramin.co.kr/zf_user/jobs/relay/view?view_type=public-recruit&rec_idx=54202283'

#### 3. 공고 ID
중복 제거에 사용하기 위함

```python

href = item.select_one(
    "div.job_tit a.str_tit"
)["href"]

rec_idx = re.search(
    r"rec_idx=(\d+)",
    href
).group(1)

```


#### 4. 직무 개수
분석용 파생 컬럼

```python
job_meta_cnt = len(job_meta)
```

#### 5. 마감 여부
향후 분석에 활용하기 위함

```python
is_closed = False

# 나중에 배치 작업 시
if "마감" in deadline:
    is_closed = True
```

#### 6. 수집 상태
ETL 운영 로그용

```python
status = "success"

# 에러 발생 시
status = "error"
```

### 1.6 추가 컬럼 반영한 데이터 수집 코드

In [10]:
def parse_job(item):

    company_nm = item.select_one(
        "div.company_nm .str_tit"
    ).get_text(strip=True)

    title_tag = item.select_one(
        "div.job_tit .str_tit"
    )

    job_tit = title_tag.get_text(strip=True)

    href = title_tag["href"]

    job_url = (
        "https://www.saramin.co.kr" + href
    )

    rec_idx = re.search(
        r"rec_idx=(\d+)",
        href
    ).group(1)

    job_meta = [
        tag.get_text(strip=True)
        for tag in item.select(
            "div.job_meta span.job_sector span"
        )
    ]

    work_place = item.select_one(
        "p.work_place"
    ).get_text(strip=True)

    career = item.select_one(
        "p.career"
    ).get_text(strip=True)

    education = item.select_one(
        "p.education"
    ).get_text(strip=True)

    salary = item.select_one(
        "p.salary"
    ).get_text(strip=True)

    deadline = item.select_one(
        "span.date"
    ).get_text(strip=True)

    registered_at = item.select_one(
        "span.deadlines"
    ).get_text(strip=True)

    return {
        "job_id": rec_idx,
        "company_nm": company_nm,
        "job_tit": job_tit,
        "job_meta": job_meta,
        "job_meta_count": len(job_meta),
        "work_place": work_place,
        "career": career,
        "education": education,
        "salary": salary,
        "deadline": deadline,
        "registered_at": registered_at,
        "job_url": job_url,
        "source": "saramin",
        "crawl_time": datetime.now().strftime("%Y-%m-%d %H:00:00"),
        "status": "success",
        "error_msg": None
    }


def fallback_job_id(item):
    """파싱 실패 시에도 list_item의 id 속성("rec-12345678")에서 job_id를 복구"""
    raw_id = item.get("id", "")
    return raw_id.replace("rec-", "") or None


def build_error_record(item, error):
    return {
        "job_id": fallback_job_id(item),
        "company_nm": None,
        "job_tit": None,
        "job_meta": None,
        "job_meta_count": None,
        "work_place": None,
        "career": None,
        "education": None,
        "salary": None,
        "deadline": None,
        "registered_at": None,
        "job_url": None,
        "source": "saramin",
        "crawl_time": datetime.now().strftime("%Y-%m-%d %H:00:00"),
        "status": "error",
        "error_msg": str(error)
    }

In [11]:
results = []

for page in range(1, 11):

    url = (
        f"https://www.saramin.co.kr/zf_user/jobs/public/list/?page={page}&isAjaxRequest=y"
    )

    print(f"크롤링 중: page={page}")

    response = requests.get(
        url,
        headers=headers
    )

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    jobs = soup.select("div.list_item")

    print(f"공고 수: {len(jobs)}")

    for item in jobs:
        try:
            results.append(
                parse_job(item)
            )
        except Exception as e:
            print(f"파싱 실패: {e}")
            results.append(
                build_error_record(item, e)
            )

df = pd.DataFrame(results)

print(f"총 수집 건수: {len(df)} (성공: {(df['status'] == 'success').sum()}, 실패: {(df['status'] == 'error').sum()})")


크롤링 중: page=1
공고 수: 20
크롤링 중: page=2
공고 수: 20
크롤링 중: page=3
공고 수: 20
크롤링 중: page=4
공고 수: 20
크롤링 중: page=5
공고 수: 20
크롤링 중: page=6
공고 수: 20
크롤링 중: page=7
공고 수: 20
크롤링 중: page=8
공고 수: 20
크롤링 중: page=9
공고 수: 20
크롤링 중: page=10
공고 수: 20
총 수집 건수: 200 (성공: 200, 실패: 0)


In [12]:
# 중복 제거
df = df.drop_duplicates(
    subset=["job_id"]
)

### 1.7 scraper.py로 Extract 모듈화 + 원시 데이터 저장

1.1 ~ 1.6에서 다듬은 파싱 로직을 `scraper.py`로 생성했다.
ETL 관점에서 `scraper.py`파일이 **Extract** 단계이며, 추후 Kafka producer가 이 모듈의 결과를 토픽에 publish하게 된다.

수집 결과(`df`)는 가공 전 원본 그대로 `data/raw/jobs_raw_<timestamp>.jsonl`에 저장한다. (재처리/디버깅 시 원본 데이터로 다시 돌아갈 수 있도록)

In [13]:
df = scrape()

print(
    f"총 수집 건수: {len(df)} "
    f"(성공: {(df['status'] == 'success').sum()}, "
    f"실패: {(df['status'] == 'error').sum()})"
)

raw_path = save_raw(df)

크롤링 중: page=1
공고 수: 20
크롤링 중: page=2
공고 수: 20
크롤링 중: page=3
공고 수: 20
크롤링 중: page=4
공고 수: 20
크롤링 중: page=5
공고 수: 20
크롤링 중: page=6
공고 수: 20
크롤링 중: page=7
공고 수: 20
크롤링 중: page=8
공고 수: 20
크롤링 중: page=9
공고 수: 20
크롤링 중: page=10
공고 수: 20
총 수집 건수: 200 (성공: 200, 실패: 0)
원시 데이터 저장: /Users/kangahhyun/Engineer/Projects/project_recruit_etl/data/raw/jobs_raw_20260617_144739.jsonl (200 rows)


### 1.8. Extract Result

In [14]:
df.head()

,job_id,company_nm,job_tit,job_meta,job_meta_count,work_place,career,education,salary,deadline,registered_at,job_url,source,crawl_time,status,error_msg
0,54204286,(주)카카오,IT 정책 및 산업 동향 연구 지원_어시스턴트,"[사무직, 서무, 문서작성, 사무보조, 사무행정]",5,경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),27분 전 등록,https://www.saramin.co.kr/zf_user/jobs/relay/v...,saramin,2026-06-17 14:00:00,success,None
1,54204194,(주)카카오,선물하기/톡딜 뷰티 CM_어시스턴트,"[이커머스, 배너광고, 배너디자인, 광고기획, 검품/검수]",5,경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),34분 전 등록,https://www.saramin.co.kr/zf_user/jobs/relay/v...,saramin,2026-06-17 14:00:00,success,None
2,54204121,(주)카카오,쇼핑하우 가격 비교 지표 관리/정산/운영_어시스턴트,"[데이터분석가, 정산관리, 요금정산, 서비스기획, 데이터분석]",5,경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),38분 전 등록,https://www.saramin.co.kr/zf_user/jobs/relay/v...,saramin,2026-06-17 14:00:00,success,None
3,54203769,네이버랩스(주),[네이버랩스] Manipulation Engineer,"[머신러닝, Ajax, C++, Python, Pytorch]",5,경기 성남시 분당구,경력무관 · 정규직,석사↑,면접 후 협의,~06.30(화),1시간 전 등록,https://www.saramin.co.kr/zf_user/jobs/relay/v...,saramin,2026-06-17 14:00:00,success,None
4,54203269,(주)지학사,2026년 6월 수시채용,"[출판기획, 교재개발, 수학, 기획]",4,서울 마포구,신입 · 경력 · 계약직,대학교(4년)↑,면접 후 협의,~07.05(일),1시간 전 등록,https://www.saramin.co.kr/zf_user/jobs/relay/v...,saramin,2026-06-17 14:00:00,success,None


## 2. Transform

Extract 단계에서 가져온 원시 데이터는 자유 텍스트(`deadline`, `registered_at`, `career`, `salary`, `work_place` 등)와 표기가 일관되지 않은 값(`(주)`, `↑`, `외` 등)이 섞여 있어 그대로는 분석에 쓸 수 없다.

원본 텍스트 컬럼은 그대로 유지한 채, 날짜/시계열 변환(2.1~2.2) → 텍스트 정제(2.3~2.5) → 구조화(2.6~2.8) → 데이터 품질 점검(2.9) 순으로 파생 컬럼을 만들고, 마지막으로 이 로직을 `transformer.py`로 모듈화해 가공 데이터를 저장한다(2.10).

### 2.1 deadline → deadline_date

수집 시점(`crawl_time`)을 기준으로 `deadline`을 실제 날짜로 변환한다.

- `내일마감` → `crawl_time` + 1일
- `오늘마감` → `crawl_time`의 날짜
- `D-N` → `crawl_time` + N일
- `~MM.DD(요일)` → 해당 연도의 MM월 DD일. 단, 그 날짜가 `crawl_time`보다 과거라면 연말을 넘긴 공고로 보고 +1년 처리

### 2.2 registered_at → registered_at_dt

수집 시점(`crawl_time`)을 기준으로 `registered_at`을 실제 datetime으로 변환한다.

- `N분 전 등록/수정` / `N시간 전 등록/수정` / `N일 전 등록/수정` → `crawl_time`에서 해당 시간만큼 역산

In [15]:
# 마감 날짜와 등록 날짜를 파싱해서 표준화된 형태로 변환하는 함수

def parse_deadline(deadline_text, reference_dt):
    if pd.isna(deadline_text):
        return None

    text = deadline_text.strip()

    if "내일마감" in text:
        return (reference_dt + timedelta(days=1)).date()

    if "오늘마감" in text or re.search(r"\d{1,2}시\s*마감", text):
        return reference_dt.date()

    m = re.match(r"D-(\d+)", text)
    if m:
        return (reference_dt + timedelta(days=int(m.group(1)))).date()

    m = re.search(r"(\d{1,2})\.(\d{1,2})", text)
    if m:
        month, day = int(m.group(1)), int(m.group(2))
        deadline_date = datetime(reference_dt.year, month, day).date()

        # 12월 공고가 1월로 넘어가는 경우 등, 과거 날짜로 계산되면 다음 해로 보정
        if deadline_date < reference_dt.date():
            deadline_date = datetime(reference_dt.year + 1, month, day).date()

        return deadline_date

    # "상시채용" 등 패턴에 안 걸리는 표기는 결측으로 남긴다
    return None


def parse_registered_at(registered_text, reference_dt):
    if pd.isna(registered_text):
        return None

    text = registered_text.strip()

    m = re.match(r"(\d+)분 전", text)
    if m:
        return reference_dt - timedelta(minutes=int(m.group(1)))

    m = re.match(r"(\d+)시간 전", text)
    if m:
        return reference_dt - timedelta(hours=int(m.group(1)))

    m = re.match(r"(\d+)일 전", text)
    if m:
        return reference_dt - timedelta(days=int(m.group(1)))

    return None


def is_deadline_closed(deadline_text, deadline_date, reference_dt):
    if deadline_date is None:
        return False

    if not pd.isna(deadline_text):
        m = re.search(r"(\d{1,2})시\s*마감", deadline_text)
        if m:
            deadline_hour = int(m.group(1))
            # "HH시 마감"은 크롤링 당일 마감이므로, 날짜가 같을 때만 시(hour)로 마감 여부를 따진다
            return reference_dt.date() == deadline_date and reference_dt.hour >= deadline_hour

    return deadline_date < reference_dt.date()

### 2.3 company_nm 정제

`(주)`, `(유)`, `(재)` 등 법인 형태를 나타내는 괄호 표기를 제거해 회사명을 통일한다 (예: `(주)국민은행` → `국민은행`, `트루넥스(주)` → `트루넥스`).

### 2.4 work_place 정제

`work_place`에 붙는 접미사 `외`를 제거한다 (예: `경기 성남시 외` → `경기 성남시`).

### 2.5 education 정제

`↑` 기호를 "이상"으로 치환하고, `대학(2,3년)이상` → `초대졸이상`, `대학교(4년)이상` → `대졸이상`으로 표준화한다.

In [16]:
def clean_company_nm(text):
    if pd.isna(text):
        return None

    text = re.sub(r"\([^)]*\)", "", text)
    text = text.replace("㈜", "")

    return text.strip()


def clean_work_place(text):
    if pd.isna(text):
        return None

    return re.sub(r"\s*외$", "", text).strip()


def clean_education(text):
    if pd.isna(text):
        return None

    text = text.replace("↑", "이상")
    text = text.replace("대학(2,3년)이상", "초대졸이상")
    text = text.replace("대학교(4년)이상", "대졸이상")

    return text

### 2.6 career → career_level, employment_type

`career`는 경력 요건(신입/경력/경력무관)과 고용형태(정규직/계약직/인턴직/교육생)가 `·`로 묶여 있다 (예: `신입 · 경력 · 정규직 외`). `·` 기준으로 split해서 `career_level`, `employment_type`을 각각 리스트로 만든다.

### 2.7 salary → salary_amount

`salary`는 `"350 만원"`처럼 숫자가 있는 경우와 `"면접 후 협의"`처럼 텍스트인 경우가 섞여 있다. 숫자가 있으면 원 단위로 환산하고(`"350 만원"` → `3,500,000`), 협의/텍스트인 경우는 `0`으로 처리한 `salary_amount` 컬럼을 만든다.

### 2.8 work_place → city, district

`work_place`(외 제거 후)를 첫 공백 기준으로 시/도(`city`)와 시/군/구(`district`)로 분리한다 (예: `경기 고양시 일산동구` → `city="경기"`, `district="고양시 일산동구"`). `"서울전체"`처럼 공백 없이 `전체`가 붙는 경우는 `city="서울"`, `district="전체"`로 분리한다.

In [17]:
CAREER_LEVELS = {"신입", "경력", "경력무관"}


def split_career(text):
    if pd.isna(text):
        return None, None

    tokens = text.split(" · ")
    levels = [t for t in tokens if t in CAREER_LEVELS]
    types = [t for t in tokens if t not in CAREER_LEVELS]

    career_level = levels if levels else None
    employment_type = types if types else None

    return career_level, employment_type


def parse_salary_amount(text):
    if pd.isna(text):
        return 0

    m = re.search(r"([\d,]+)\s*만원", text)
    if not m:
        return 0

    return int(m.group(1).replace(",", "")) * 10000


def split_work_place(text):
    if pd.isna(text):
        return None, None

    if "전체" in text and " " not in text:
        return text.replace("전체", ""), "전체"

    tokens = text.split(" ", 1)
    city = tokens[0]
    district = tokens[1] if len(tokens) > 1 else None

    return city, district

In [18]:
df["crawl_time"] = pd.to_datetime(df["crawl_time"])

df["deadline_date"] = df.apply(
    lambda row: parse_deadline(row["deadline"], row["crawl_time"]),
    axis=1
)

df["registered_at_dt"] = df.apply(
    lambda row: parse_registered_at(row["registered_at"], row["crawl_time"]),
    axis=1
)

df["company_nm"] = df["company_nm"].apply(clean_company_nm)
df["work_place"] = df["work_place"].apply(clean_work_place)
df["career"] = df["career"].apply(clean_work_place)
df["education"] = df["education"].apply(clean_education)

# 크롤링 시점(crawl_time) 기준으로 마감 여부 판단 ("HH시 마감"은 날짜가 같으면 시(hour)까지 비교)
df["is_closed"] = df.apply(
    lambda row: is_deadline_closed(row["deadline"], row["deadline_date"], row["crawl_time"]),
    axis=1
)

df[["career_level", "employment_type"]] = df["career"].apply(
    lambda x: pd.Series(split_career(x))
)

df["salary_amount"] = df["salary"].apply(parse_salary_amount)

df[["city", "district"]] = df["work_place"].apply(
    lambda x: pd.Series(split_work_place(x))
)

print("deadline 변환 실패 건수:", df["deadline_date"].isna().sum())
print("registered_at 변환 실패 건수:", df["registered_at_dt"].isna().sum())

df[[
    "company_nm", "work_place", "city", "district", "education",
    "career", "career_level", "employment_type",
    "salary", "salary_amount",
    "deadline", "deadline_date",
    "registered_at", "registered_at_dt",
    "is_closed"
]].head(10)

deadline 변환 실패 건수: 0
registered_at 변환 실패 건수: 0


,company_nm,work_place,city,district,education,career,career_level,employment_type,salary,salary_amount,deadline,deadline_date,registered_at,registered_at_dt,is_closed
0,카카오,경기 성남시 분당구,경기,성남시 분당구,학력무관,경력무관 · 인턴직,[경력무관],[인턴직],면접 후 협의,0,~06.30(화),2026-06-30,27분 전 등록,2026-06-17 13:33:00,False
1,카카오,경기 성남시 분당구,경기,성남시 분당구,학력무관,경력무관 · 인턴직,[경력무관],[인턴직],면접 후 협의,0,~06.30(화),2026-06-30,34분 전 등록,2026-06-17 13:26:00,False
2,카카오,경기 성남시 분당구,경기,성남시 분당구,학력무관,경력무관 · 인턴직,[경력무관],[인턴직],면접 후 협의,0,~06.30(화),2026-06-30,38분 전 등록,2026-06-17 13:22:00,False
3,네이버랩스,경기 성남시 분당구,경기,성남시 분당구,석사이상,경력무관 · 정규직,[경력무관],[정규직],면접 후 협의,0,~06.30(화),2026-06-30,1시간 전 등록,2026-06-17 13:00:00,False
4,지학사,서울 마포구,서울,마포구,대졸이상,신입 · 경력 · 계약직,"[신입, 경력]",[계약직],면접 후 협의,0,~07.05(일),2026-07-05,1시간 전 등록,2026-06-17 13:00:00,False
5,SK텔레콤,서울 중구,서울,중구,학력무관,신입 · 계약직,[신입],[계약직],면접 후 협의,0,D-4,2026-06-21,1시간 전 등록,2026-06-17 13:00:00,False
6,경북대학교병원,대구 북구,대구,북구,초대졸이상,신입 · 경력 · 계약직,"[신입, 경력]",[계약직],면접 후 협의,0,D-5,2026-06-22,2시간 전 등록,2026-06-17 12:00:00,False
7,경북대학교병원,대구 북구,대구,북구,초대졸이상,신입 · 경력 · 계약직,"[신입, 경력]",[계약직],면접 후 협의,0,D-5,2026-06-22,2시간 전 등록,2026-06-17 12:00:00,False
8,에스케이하이닉스,경기 성남시 분당구,경기,성남시 분당구,학력무관,신입 · 정규직,[신입],[정규직],면접 후 협의,0,D-6,2026-06-23,2시간 전 등록,2026-06-17 12:00:00,False
9,에스케이하이닉스,경기 이천시,경기,이천시,학력무관,신입 · 정규직,[신입],[정규직],면접 후 협의,0,D-6,2026-06-23,2시간 전 등록,2026-06-17 12:00:00,False


### 2.9 결측치 / 이상치 점검

변환이 끝난 직후 데이터 품질을 확인한다.
- 컬럼별 결측치 개수
- `salary_amount`가 0인데 실제로는 "협의"류 표현이 아닌 경우(파싱이 안 된 이상치 후보)
- `career_level` / `employment_type` / `district` 분리 실패 건수

In [19]:
print("=== 컬럼별 결측치 개수 ===")
print(df.isna().sum())

salary_amount_anomaly = df[
    (df["salary_amount"] == 0) & (~df["salary"].str.contains("협의|채용시", na=False))
]

print("\n=== 이상치 점검 ===")
print("salary_amount=0 인데 '협의'류 표현이 아닌 행:", len(salary_amount_anomaly))
print("career_level 분리 실패:", df["career_level"].isna().sum())
print("employment_type 분리 실패:", df["employment_type"].isna().sum())
print("district 분리 실패:", df["district"].isna().sum())

=== 컬럼별 결측치 개수 ===
job_id                0
company_nm            0
job_tit               0
job_meta              0
job_meta_count        0
work_place            0
career                0
education             0
salary                0
deadline              0
registered_at         0
job_url               0
source                0
crawl_time            0
status                0
error_msg           200
deadline_date         0
registered_at_dt      0
is_closed             0
career_level          0
employment_type       0
salary_amount         0
city                  0
district              2
dtype: int64

=== 이상치 점검 ===
salary_amount=0 인데 '협의'류 표현이 아닌 행: 0
career_level 분리 실패: 0
employment_type 분리 실패: 0
district 분리 실패: 2


In [20]:
df

,job_id,company_nm,job_tit,job_meta,job_meta_count,work_place,career,education,salary,deadline,...,status,error_msg,deadline_date,registered_at_dt,is_closed,career_level,employment_type,salary_amount,city,district
0,54204286,카카오,IT 정책 및 산업 동향 연구 지원_어시스턴트,"[사무직, 서무, 문서작성, 사무보조, 사무행정]",5,경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),...,success,None,2026-06-30,2026-06-17 13:33:00,False,[경력무관],[인턴직],0,경기,성남시 분당구
1,54204194,카카오,선물하기/톡딜 뷰티 CM_어시스턴트,"[이커머스, 배너광고, 배너디자인, 광고기획, 검품/검수]",5,경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),...,success,None,2026-06-30,2026-06-17 13:26:00,False,[경력무관],[인턴직],0,경기,성남시 분당구
2,54204121,카카오,쇼핑하우 가격 비교 지표 관리/정산/운영_어시스턴트,"[데이터분석가, 정산관리, 요금정산, 서비스기획, 데이터분석]",5,경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),...,success,None,2026-06-30,2026-06-17 13:22:00,False,[경력무관],[인턴직],0,경기,성남시 분당구
3,54203769,네이버랩스,[네이버랩스] Manipulation Engineer,"[머신러닝, Ajax, C++, Python, Pytorch]",5,경기 성남시 분당구,경력무관 · 정규직,석사이상,면접 후 협의,~06.30(화),...,success,None,2026-06-30,2026-06-17 13:00:00,False,[경력무관],[정규직],0,경기,성남시 분당구
4,54203269,지학사,2026년 6월 수시채용,"[출판기획, 교재개발, 수학, 기획]",4,서울 마포구,신입 · 경력 · 계약직,대졸이상,면접 후 협의,~07.05(일),...,success,None,2026-07-05,2026-06-17 13:00:00,False,"[신입, 경력]",[계약직],0,서울,마포구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,54174021,하이윈코퍼레이션,[HIWIN] 수원공장/창원공장 인재 모집,"[전동지게차, 지게차, 기계조작원, 단순생산직, 생산관리]",5,경기 수원시 권선구,신입 · 경력 · 정규직,고졸이상,면접 후 협의,~06.30(화),...,success,None,2026-06-30,2026-06-15 14:00:00,False,"[신입, 경력]",[정규직],0,경기,수원시 권선구
196,54173709,에이쓰리,[대기업생산직]삼성1차협력 대덕전자 반도체 PCB 부문 생산 신입 대규모 채용 기숙사有,"[기계조작원, 단순생산직, 생산관리, 설비OP, 외관검사원]",5,경기 안산시,경력무관 · 계약직,학력무관,면접 후 협의,D-7,...,success,None,2026-06-24,2026-06-15 14:00:00,False,[경력무관],[계약직],0,경기,안산시
197,54173551,건축공간연구원,위촉직 채용공고(6월-에너지 소비 총량제 운영모델 구축),"[연구원, 정책연구, 환경오염, 에너지관리, 법무]",5,세종 세종특별자치시,경력무관 · 계약직,석사이상,면접 후 협의,~06.29(월),...,success,None,2026-06-29,2026-06-15 14:00:00,False,[경력무관],[계약직],0,세종,세종특별자치시
198,54173420,이엠에스,주간/주간2교대 대기업 사내협력사 정규직 사원 모집,"[단순생산직, 생산관리, 제조가공, 3교대, 주간근무]",5,경북 구미시,경력무관 · 정규직,고졸이상,면접 후 협의,D-2,...,success,None,2026-06-19,2026-06-15 14:00:00,False,[경력무관],[정규직],0,경북,구미시


### 2.10 transformer.py로 Transform 모듈화 + 가공 데이터 저장

2.1 ~ 2.9에서 다듬은 변환/정제 로직을 `transformer.py`로 옮겼다.
ETL 관점에서 이 파일이 **Transform** 단계이며, 추후 ETL Worker가 이 모듈을 호출해 Elasticsearch에 적재(Load)하게 된다.

변환 결과는 1.7의 raw 데이터 저장과 같은 방식으로 `data/processed/jobs_processed_<timestamp>.jsonl`에 저장한다.

In [21]:
raw_df = scrape()

df = transform(raw_df)
log_data_quality(df)

processed_path = save_processed(df)

df.head()

크롤링 중: page=1
공고 수: 20
크롤링 중: page=2
공고 수: 20
크롤링 중: page=3
공고 수: 20
크롤링 중: page=4
공고 수: 20
크롤링 중: page=5
공고 수: 20
크롤링 중: page=6
공고 수: 20
크롤링 중: page=7
공고 수: 20
크롤링 중: page=8
공고 수: 20
크롤링 중: page=9
공고 수: 20
크롤링 중: page=10
공고 수: 20
=== 컬럼별 결측치 개수 ===
job_id                0
company_nm            0
job_tit               0
job_meta              0
job_meta_count        0
work_place            0
career                0
education             0
salary                0
deadline              0
registered_at         0
job_url               0
source                0
crawl_time            0
status                0
error_msg           200
deadline_date         0
registered_at_dt      0
is_closed             0
career_level          0
employment_type       0
salary_amount         0
city                  0
district              2
dtype: int64

=== 이상치 점검 ===
salary_amount=0 인데 '협의'류 표현이 아닌 행: 0
career_level 분리 실패: 0
employment_type 분리 실패: 0
district 분리 실패: 2
가공 데이터 저장: /Users/kangahhyun/Engineer/Proje

,job_id,company_nm,job_tit,job_meta,job_meta_count,work_place,career,education,salary,deadline,...,status,error_msg,deadline_date,registered_at_dt,is_closed,career_level,employment_type,salary_amount,city,district
0,54204286,카카오,IT 정책 및 산업 동향 연구 지원_어시스턴트,"[사무직, 서무, 문서작성, 사무보조, 사무행정]",5,경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),...,success,None,2026-06-30,2026-06-17 13:33:00,False,[경력무관],[인턴직],0,경기,성남시 분당구
1,54204194,카카오,선물하기/톡딜 뷰티 CM_어시스턴트,"[이커머스, 배너광고, 배너디자인, 광고기획, 검품/검수]",5,경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),...,success,None,2026-06-30,2026-06-17 13:26:00,False,[경력무관],[인턴직],0,경기,성남시 분당구
2,54204121,카카오,쇼핑하우 가격 비교 지표 관리/정산/운영_어시스턴트,"[데이터분석가, 정산관리, 요금정산, 서비스기획, 데이터분석]",5,경기 성남시 분당구,경력무관 · 인턴직,학력무관,면접 후 협의,~06.30(화),...,success,None,2026-06-30,2026-06-17 13:22:00,False,[경력무관],[인턴직],0,경기,성남시 분당구
3,54203769,네이버랩스,[네이버랩스] Manipulation Engineer,"[머신러닝, Ajax, C++, Python, Pytorch]",5,경기 성남시 분당구,경력무관 · 정규직,석사이상,면접 후 협의,~06.30(화),...,success,None,2026-06-30,2026-06-17 13:00:00,False,[경력무관],[정규직],0,경기,성남시 분당구
4,54203269,지학사,2026년 6월 수시채용,"[출판기획, 교재개발, 수학, 기획]",4,서울 마포구,신입 · 경력 · 계약직,대졸이상,면접 후 협의,~07.05(일),...,success,None,2026-07-05,2026-06-17 13:00:00,False,"[신입, 경력]",[계약직],0,서울,마포구


## 3. Kafka

Kafka는 **Producer(발행자)** 와 **Consumer(소비자)** 사이에서 메시지를 중개하는 분산 스트리밍 플랫폼이다. 이 파이프라인에서는 스크래퍼(Producer)와 ETL Worker(Consumer)를 분리해, 어느 한쪽이 느리거나 다운돼도 데이터가 유실되지 않게 한다.

| 문제 | Kafka 없이 | Kafka 있을 때 |
|------|-----------|-------------|
| 속도 차이 | 크롤러 속도에 ETL이 맞춰야 함 | 각자 속도로 독립 처리 |
| 장애 격리 | ES가 다운되면 수집도 중단 | Kafka가 메시지 보존 → 복구 후 이어서 처리 |
| 확장성 | 단일 프로세스에 종속 | Worker 인스턴스 수평 확장 가능 |

**파이프라인 흐름**

```
[cron 11:00, 18:00]
        ↓
kafka_producer.py   →   scrape() → save_raw()
                    →   Kafka topic "saramin-jobs-raw" 발행
                                ↓ (etl_worker가 구독 중)
                    etl_worker.py (상시 실행)
                    → 30초 타임아웃 배치 flush
                    → transform() → save_processed() → load()
                                                         ↓
                                                  Elasticsearch
```

### 3.1 Kafka 구성

`docker-compose.yml`에 `apache/kafka:3.7.0` 서비스를 추가해 **KRaft 모드**(Zookeeper 없음)로 실행한다.

```bash
# Kafka 컨테이너 실행
docker compose up -d kafka

# topic 생성 (최초 1회)
docker exec recruit-kafka /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create --topic saramin-jobs-raw \
  --partitions 1 --replication-factor 1
```

### 3.2 kafka_producer.py

`scraper.py`의 `scrape()`를 호출해 수집한 뒤, 각 공고 딕셔너리를 JSON으로 직렬화해 `saramin-jobs-raw` 토픽에 publish하는 모듈이다. cron이 11:00, 18:00 두 차례 이 파일을 실행한다.

### 3.3 etl_worker.py

`saramin-jobs-raw` 토픽을 구독하는 장기 실행 프로세스다. 5초 간격으로 폴링해 메시지를 누적하고, 마지막 메시지 수신 후 30초가 지나면 배치를 `transform() → save_processed() → load()` 순으로 처리한다.

```bash
# 별도 터미널에서 상시 실행
python etl_worker.py
```

### 3.4 cron 스케줄링

macOS 기본 스케줄러(`crontab`)로 `kafka_producer.py`를 하루 2회 자동 실행한다.

```crontab
0 11 * * * cd /Users/kangahhyun/Engineer/Projects/project_recruit_etl && /opt/anaconda3/bin/python3 kafka_producer.py >> logs/cron.log 2>&1
0 18 * * * cd /Users/kangahhyun/Engineer/Projects/project_recruit_etl && /opt/anaconda3/bin/python3 kafka_producer.py >> logs/cron.log 2>&1
```

In [ ]:
produce()

![kafka_producer 실행 결과](images/kafka_producer_terminal.png)

*kafka_producer.py 실행 화면: 사람인 10페이지(196건) 크롤링 후 saramin-jobs-raw 토픽에 발행 완료*

![etl_worker 실행 결과](images/etl_worker_terminal.png)

*etl_worker.py 실행 화면: 200건 메시지 누적 후 Transform → 결측치/이상치 점검 → Load 배치 처리 완료*

## 4. Load

가공 데이터를 **Elasticsearch 인덱스**에 적재하고, **Kibana**로 대시보드를 구성한다.

### 4.1 인덱스 설계

Transform 단계에서 만든 파생 컬럼 위주로 색인하고, 원본 텍스트 컬럼(`career`, `registered_at`)과 운영용 컬럼(`job_meta_count`, `error_msg`)은 제외한다.

- **`job_id` → ES `_id`**: 공고를 재수집해 같은 `job_id`가 들어와도 upsert로 처리되어 중복 색인을 방지한다.
- **`career_level`, `employment_type`, `job_meta`**: `keyword` 배열 타입. ES가 각 요소를 독립적으로 역색인하므로 리스트 전체가 아닌 요소 단위로 terms 집계가 가능하다.
- **`company_nm`**: `text` + `keyword` 멀티필드. 전문(full-text) 검색과 terms 집계를 동시에 지원한다.
- **날짜 컬럼** (`deadline_date`, `registered_at_dt`, `crawl_time`): `date` 타입. 범위 필터와 date histogram에 활용한다.

### 4.2 loader.py

`transformer.py`와 동일한 구조로 `loader.py`를 생성했다. ETL 관점에서 이 파일이 **Load** 단계이며, `_to_actions()`가 DataFrame 행을 ES bulk action으로 변환하고, `bulk()` API로 한 번에 색인한다.

In [ ]:
print(f"인덱스명: {INDEX_NAME}")
print(f"\n색인 컬럼 ({len(LOAD_COLUMNS)}개):")
for col in LOAD_COLUMNS:
    dtype = MAPPING["mappings"]["properties"].get(col, {}).get("type", "-")
    print(f"  {col}: {dtype}")

In [ ]:
success, errors = load(df)

### 4.3 Kibana Dashboard

Kibana Data View(`saramin-jobs`)를 생성한 뒤 아래 8개 패널로 대시보드를 구성했다.

| # | 패널 | 차트 타입 | 기준 필드 | 인사이트 |
|---|------|-----------|----------|---------|
| 1 | 총 공고 / 신입 가능 / 수도권 공고 | Metric | `career_level`, `city` | 전체 현황 한눈에 파악 |
| 2 | 지역별 공고 수 | Horizontal bar | `city` | 채용 밀집 지역 식별 |
| 3 | 경력 수준 × 고용형태 | Stacked bar | `career_level` + `employment_type` | 신입 정규직 비중 비교 |
| 4 | 인기 직무 태그 Top 10 | Horizontal bar | `job_meta` | 수요 높은 직무 파악 |
| 5 | 학력 요건 분포 | Donut | `education` | 학력 무관 vs 제한 공고 비율 |
| 6 | 급여 분포 | Histogram | `salary_amount` > 0 | 급여 명시 공고의 연봉대 분포 |
| 7 | 마감일별 공고 수 | Bar | `deadline_date` | 마감 임박 공고 현황 |
| 8 | 공고 목록 | Data table | 전체 필드 | 상세 탐색 |

`job_meta`는 배열 타입이지만 ES가 각 요소를 개별적으로 역색인하므로, terms 집계 시 리스트 전체가 아닌 **태그 단위로 카운팅**된다.

**대시보드**: [Kibana Dashboard](http://localhost:5601/app/dashboards#/view/88ed19bf-61f6-4c08-bce4-4d3979c10912?_g=(refreshInterval:(pause:!t,value:60000),time:(from:'2026-06-17T15:00:00.000Z',to:'2026-06-17T15:00:00.000Z'))&_a=())

**공고 목록 Discover**: [Kibana Discover](http://localhost:5601/app/r?l=DISCOVER_APP_LOCATOR&v=8.13.0&lz=N4IgjgrgpgTgniAXKSsGJCANCANgQwDsBzCfYqJEAa2nhAF8cBnAexgBckBtbkAYxj4A7rgD6HAJYBbSjgAmUZvxABdVTn6tcEaYWY8B%2BGFFhjcUAG5Rc2EFGkAHXKzizCHCXEdyQAK1YAIwlJLhwoeQh%2BfClWQjstJyI4MUJpBNCEBSh8eVxJQigxeRjKDRACxQAPKgBGQIAzQPkATihagFoGgDYAVl6OgBYoQYB2DsDuwIAmDt75Xv5A2oBmflHFhrsCjlhLfFsMfAgOVjsGyVxdmANEbnKAC0lFAGEH4y5EBoPmKBwYVjCAASUEkxAenxWOAeOUUMAASoCQWCIUgoSBiDBnkhgEwQMx8NZ5ABlHIwfgPACS8ioAA5%2BPIxr0oN0Ot1%2BNNZmNRqyWvIAAxQNm9fmBQZLeQNUYrWl2EocfAANUkUGE1LqjWabU6PX6QxG40mMzmCyWq3WmzsUlk8KIFBxIAaAPSGGm%2FOmrP5rNqowAKrVeoh%2Bfyg%2FyAHTB%2FkALStZ1d7s93r9AdDoYjwZjeJMTqUVI8ewODv2OkoiG6kf5OEcx1%2BSA4MGgDAYQA%3D)

![Kibana Dashboard](images/kibana_dashboard.png)

*Kibana 대시보드: 경력별 공고 수, 지역별 공고 수, 경력별 고용형태, 인기 직무 태그, 학력 조건별 고용 형태, 마감일별 공고 수 6개 패널*

![Kibana 공고 목록 테이블](images/kibana_table.png)

*Kibana Discover 공고 목록 테이블: crawl_time, career_level, employment_type, job_tit, education, company_nm, city, deadline_date 8개 컬럼으로 공고 상세 탐색*